<a href="https://colab.research.google.com/github/ksmith2001/Cardiothoracic-Ratio-Ai-V2/blob/main/CTR__model_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO


In [ ]:
import yaml

yaml_path = '/content/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content'
data['train'] = 'train/images'
data['val'] = 'valid/images'
if 'test' in data:
    data['test'] = 'test/images'


with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml successfully configured for the root folder!")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")


model.train(
    data="/content/data.yaml",
    epochs=50,
    imgsz=640,
    device=0
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import io
from google.colab import files
from ultralytics import YOLO

# 1. Update path directly to your train-3 weights folder
MODEL_PATH = "/content/runs/detect/train-3/weights/best.pt"

def run_ctr_interactive_uploader():
    print("=== CTR INTERACTIVE ANALYSIS APP ===")

    # Verify the model path exists first
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Could not find model weights at: {MODEL_PATH}")
        print("Please check your file sidebar to verify if it is named train-3 or something else.")
        return

    # 2. Trigger the native Google Colab file upload widget button
    print("👇 Click the button below to upload your Chest X-Ray image:")
    uploaded = files.upload()

    if not uploaded:
        print("❌ Upload canceled.")
        return

    # Get the file name of the uploaded image
    image_name = list(uploaded.keys())[0]
    image_path = os.path.join('/content', image_name)

    print(f"\n🧠 Loading model and processing: {image_name}...")

    # 3. Initialize model and run inference
    model = YOLO(MODEL_PATH)
    results = model.predict(source=image_path, conf=0.25, verbose=False)

    # Load image for overlay formatting
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heart_width = None
    thoracic_width = None

    # 4. Extract bounding box dimensions
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        xyxy = box.xyxy[0].tolist()  # [xmin, ymin, xmax, ymax]

        width = xyxy[2] - xyxy[0]   # Horizontal width delta calculation
        class_name = model.names[cls_id].lower()

        if "heart" in class_name:
            heart_width = width
            cv2.rectangle(img, (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3])), (0, 122, 255), 5)
        elif "thoracic" in class_name or "cavity" in class_name or "chest" in class_name:
            thoracic_width = width
            cv2.rectangle(img, (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3])), (52, 199, 89), 5)

    # 5. Math & Diagnostic Metric Printing
    print("\n" + "="*40)
    print("📊 CTR DIAGNOSTIC REPORT")
    print("="*40)

    if heart_width and thoracic_width:
        ctr = heart_width / thoracic_width
        result_text = f"Calculated CTR: {ctr:.2f}"

        print(f" 🫁 Thoracic Width : {thoracic_width:.1f} pixels")
        print(f" 💓 Heart Width    : {heart_width:.1f} pixels")
        print(f" 📈 Ratio Result   : {ctr:.2f}")

        if ctr > 0.50:
            status = "CARDIOMEGALY DETECTED"
            text_color = (255, 59, 48)  # Warning Red
            print(f" 🚨 Status Flag    : {status}")
        else:
            status = "NORMAL HEART SIZE"
            text_color = (52, 199, 89)  # Healthy Green
            print(f" ✅ Status Flag    : {status}")

        # Write results text directly onto image canvas layout
        h, w, _ = img.shape
        cv2.putText(img, result_text, (40, int(h * 0.06)), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)
        cv2.putText(img, status, (40, int(h * 0.12)), cv2.FONT_HERSHEY_SIMPLEX, 1.5, text_color, 4)
    else:
        print("❌ Error: The model couldn't isolate BOTH the heart and thoracic cavity.")
        print("Tip: Make sure the X-Ray is an un-cropped, standard frontal PA view.")
    print("="*40 + "\n")

    # 6. Render the visual display
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Execute the uploader workspace
run_ctr_interactive_uploader()